In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
!pip install imblearn
from imblearn.over_sampling import SMOTE
from random import shuffle
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier

<h1>Data Loading</h1>

In [ ]:
data = pd.read_csv("/kaggle/input/amex-competition/dataset/train.csv")
data

<h1>Data Engineering</h1>

In [ ]:
data.isna().sum()

<p>Check the dataset distribution and replace with mean or median</p>

In [ ]:
data.no_of_children.hist()
data.no_of_children.fillna(np.median(data.no_of_children.dropna()), inplace=True)

In [ ]:
data.no_of_days_employed.hist()
data.no_of_days_employed.fillna(np.median(data.no_of_days_employed.dropna()), inplace=True)

In [ ]:
data.total_family_members.hist()
data.total_family_members.fillna(np.median(data.total_family_members.dropna()), inplace=True)

In [ ]:
data.yearly_debt_payments.hist()
data.yearly_debt_payments.fillna(np.median(data.yearly_debt_payments.dropna()), inplace=True)

In [ ]:
data.credit_score.hist()
data.credit_score.fillna(np.median(data.credit_score.dropna()), inplace=True)

In [ ]:
data.isna().sum()

In [ ]:
data.owns_car.fillna(data.owns_car.mode().values[0],inplace=True)
data.migrant_worker.fillna(data.migrant_worker.mode().values[0],inplace=True)

<h2>Synthetic features</h2>
<p>Generating meaningful synthetic features from the data</p>

In [ ]:
def get_profit_category(value):
    if value < 0 :
        return "Loss"
    elif value == 0 :
        return "Zero"
    else:
        return "Profit"

data["in_hand_balance"] = (data.net_yearly_income - data.yearly_debt_payments)
data["in_profit"] = [get_profit_category(i) for i in data["in_hand_balance"].values]
data["total_income_received"] = data.net_yearly_income * data.no_of_days_employed
data["employment_years"] = (data.no_of_days_employed / 365.0)

In [ ]:
data.employment_years.clip(upper = np.median(data.employment_years), inplace=True)

<h1>Exploratory Data Analysis</h1>

In [ ]:
cnames = {
'antiquewhite':         '#FAEBD7',
'aqua':                 '#00FFFF',
'aquamarine':           '#7FFFD4',
'beige':                '#F5F5DC',
'bisque':               '#FFE4C4',
'black':                '#000000',
'blanchedalmond':       '#FFEBCD',
'blue':                 '#0000FF',
'blueviolet':           '#8A2BE2',
'brown':                '#A52A2A',
'burlywood':            '#DEB887',
'cadetblue':            '#5F9EA0',
'chartreuse':           '#7FFF00',
'chocolate':            '#D2691E',
'coral':                '#FF7F50',
'cornflowerblue':       '#6495ED',
'crimson':              '#DC143C',
'cyan':                 '#00FFFF',
'darkblue':             '#00008B',
'darkcyan':             '#008B8B',
'darkgoldenrod':        '#B8860B',
#'darkgray':             '#A9A9A9',
'darkgreen':            '#006400',
'darkkhaki':            '#BDB76B',
'darkmagenta':          '#8B008B',
'darkolivegreen':       '#556B2F',
'darkorange':           '#FF8C00',
'darkorchid':           '#9932CC',
'darkred':              '#8B0000',
'darksalmon':           '#E9967A',
'darkseagreen':         '#8FBC8F',
'darkslateblue':        '#483D8B',
#'darkslategray':        '#2F4F4F',
'darkturquoise':        '#00CED1',
'darkviolet':           '#9400D3',
'deeppink':             '#FF1493',
'deepskyblue':          '#00BFFF',
#'dimgray':              '#696969',
'dodgerblue':           '#1E90FF',
'firebrick':            '#B22222',
'forestgreen':          '#228B22',
'fuchsia':              '#FF00FF',
'gainsboro':            '#DCDCDC',
'gold':                 '#FFD700',
'goldenrod':            '#DAA520',
#'gray':                 '#808080',
'green':                '#008000',
'greenyellow':          '#ADFF2F',
'honeydew':             '#F0FFF0',
'hotpink':              '#FF69B4',
'indianred':            '#CD5C5C',
'indigo':               '#4B0082',
'ivory':                '#FFFFF0',
'khaki':                '#F0E68C',
'lavender':             '#E6E6FA',
'lawngreen':            '#7CFC00',
'lemonchiffon':         '#FFFACD',
'lightblue':            '#ADD8E6',
'lightcoral':           '#F08080',
'lightcyan':            '#E0FFFF',
'lightgoldenrodyellow': '#FAFAD2',
'lightgreen':           '#90EE90',
#'lightgray':            '#D3D3D3',
'lightpink':            '#FFB6C1',
'lightsalmon':          '#FFA07A',
'lightseagreen':        '#20B2AA',
'lightskyblue':         '#87CEFA',
#'lightslategray':       '#778899',
'lightsteelblue':       '#B0C4DE',
'lime':                 '#00FF00',
'limegreen':            '#32CD32',
'magenta':              '#FF00FF',
'maroon':               '#800000',
'mediumaquamarine':     '#66CDAA',
'mediumblue':           '#0000CD',
'mediumorchid':         '#BA55D3',
'mediumpurple':         '#9370DB',
'mediumseagreen':       '#3CB371',
'mediumslateblue':      '#7B68EE',
'mediumspringgreen':    '#00FA9A',
'mediumturquoise':      '#48D1CC',
'mediumvioletred':      '#C71585',
'midnightblue':         '#191970',
'mintcream':            '#F5FFFA',
'moccasin':             '#FFE4B5',
'navajowhite':          '#FFDEAD',
'navy':                 '#000080',
'olive':                '#808000',
'olivedrab':            '#6B8E23',
'orange':               '#FFA500',
'orangered':            '#FF4500',
'orchid':               '#DA70D6',
'palegoldenrod':        '#EEE8AA',
'palegreen':            '#98FB98',
'paleturquoise':        '#AFEEEE',
'palevioletred':        '#DB7093',
'papayawhip':           '#FFEFD5',
'peachpuff':            '#FFDAB9',
'peru':                 '#CD853F',
'pink':                 '#FFC0CB',
'plum':                 '#DDA0DD',
'powderblue':           '#B0E0E6',
'purple':               '#800080',
'red':                  '#FF0000',
'rosybrown':            '#BC8F8F',
'royalblue':            '#4169E1',
'saddlebrown':          '#8B4513',
'salmon':               '#FA8072',
'sandybrown':           '#FAA460',
'seagreen':             '#2E8B57',
'sienna':               '#A0522D',
'silver':               '#C0C0C0',
'skyblue':              '#87CEEB',
'slateblue':            '#6A5ACD',
#'slategray':            '#708090',
'springgreen':          '#00FF7F',
'steelblue':            '#4682B4',
'tan':                  '#D2B48C',
'teal':                 '#008080',
'thistle':              '#D8BFD8',
'tomato':               '#FF6347',
'turquoise':            '#40E0D0',
'violet':               '#EE82EE',
'wheat':                '#F5DEB3',
'white':                '#FFFFFF',
'yellow':               '#FFFF00',
'yellowgreen':          '#9ACD32'}

colors = list(cnames.keys())

In [ ]:
data.columns

<h1>How old are they?</h1>

In [ ]:
plt.figure(figsize=(10,8))
shuffle(colors)
plt.subplot(2,2,1)
plt.hist(data.age,color=colors[0])
plt.xlabel("Age group")
plt.ylabel("Count")
plt.title("Which age group is dominant?")

shuffle(colors)
plt.subplot(2,2,2)
plt.hist(data[data.credit_card_default == 0].age,color=colors[0])
plt.xlabel("Age group")
plt.ylabel("Count")
plt.title("Which age group is dominant among the non-defaulters?")

shuffle(colors)
plt.subplot(2,2,3)
plt.hist(data[data.credit_card_default == 1].age,color=colors[0])
plt.xlabel("Age group")
plt.ylabel("Count")
plt.title("Which age group is dominant among the defaulters?")
plt.tight_layout()

<p><b>Inference : </b>Among both classes, <b>25</b> <b>40</b> and <b>55</b> tend to be the dominant ages.</p>

<h1>Which gender is dominant?</h1>

In [ ]:
plt.figure(figsize=(10,9))
plt.subplot(2,2,1)
shuffle(colors)
un, count = np.unique(data.gender, return_counts=True)
plt.bar(un, count, color=colors)
plt.xlabel("Gender")
plt.ylabel("Count")
plt.title("Which gender group is dominant?")

plt.subplot(2,2,2)
shuffle(colors)
un, count = np.unique(data[data.credit_card_default == 1].gender, return_counts=True)
plt.bar(un, count, color=colors)
plt.xlabel("Gender")
plt.ylabel("Count")
plt.title("Which gender group is dominant among the defaulters?")

plt.subplot(2,2,3)
shuffle(colors)
un, count = np.unique(data[data.credit_card_default == 0].gender, return_counts=True)
plt.bar(un, count, color=colors)
plt.xlabel("Gender")
plt.ylabel("Count")
plt.title("Which gender group is dominant among the non-defaulters?")
plt.tight_layout()

<p><b>Inference : </b><b>Females</b> tend to be more <b>Non-defaulters</b> while <b>Males</b> tend to be more in number in <b>Defaulters</b>.</p>

<h1>How many owns a vehicle?</h1>

In [ ]:
plt.figure(figsize=(10,9))
plt.subplot(2,2,1)
shuffle(colors)
un, count = np.unique(data.owns_car.dropna(), return_counts=True)
un = ["Yes" if i.lower() == "y" else "No" for i in un]
plt.bar(un, count, color=colors)
plt.xlabel("Vehicle Ownership")
plt.ylabel("Count")
plt.title("How many own a car?")

plt.subplot(2,2,2)
shuffle(colors)
un, count = np.unique(data[data.credit_card_default == 1].owns_car.dropna(), return_counts=True)
un = ["Yes" if i.lower() == "y" else "No" for i in un]
plt.bar(un, count, color=colors)
plt.xlabel("Vehicle Ownership")
plt.ylabel("Count")
plt.title("How many own a car among the defaulters?")

plt.subplot(2,2,3)
shuffle(colors)
un, count = np.unique(data[data.credit_card_default == 0].owns_car.dropna(), return_counts=True)
un = ["Yes" if i.lower() == "y" else "No" for i in un]
plt.bar(un, count, color=colors)
plt.xlabel("Vehicle Ownership")
plt.ylabel("Count")
plt.title("How many own a car among the non-defaulters?")
plt.tight_layout()

<p><b>Inference : </b>Among both classes, majority do not <b>own</b> a car.</p>

<h1>How many have a residence ownership?</h1>

In [ ]:
plt.figure(figsize=(10,9))
plt.subplot(2,2,1)
shuffle(colors)
un, count = np.unique(data.owns_house.dropna(), return_counts=True)
un = ["Yes" if i.lower() == "y" else "No" for i in un]
plt.bar(un, count, color=colors)
plt.xlabel("House Ownership")
plt.ylabel("Count")
plt.title("How many owns a residence?")

plt.subplot(2,2,2)
shuffle(colors)
un, count = np.unique(data[data.credit_card_default == 1].owns_house.dropna(), return_counts=True)
un = ["Yes" if i.lower() == "y" else "No" for i in un]
plt.bar(un, count, color=colors)
plt.xlabel("House Ownership")
plt.ylabel("Count")
plt.title("How many owns a residence among the defaulters?")

plt.subplot(2,2,3)
shuffle(colors)
un, count = np.unique(data[data.credit_card_default == 0].owns_house.dropna(), return_counts=True)
un = ["Yes" if i.lower() == "y" else "No" for i in un]
plt.bar(un, count, color=colors)
plt.xlabel("House Ownership")
plt.ylabel("Count")
plt.title("How many owns a residence among the non-defaulters?")
plt.tight_layout()

<p><b>Inference : </b>Majority <b>Own</b> a residence among the defaulters while the opposite happens in the case of <b>Non-defaulters</b>.</p>

<h1>How many children do they have?</h1>

In [ ]:
plt.figure(figsize=(10,9))
plt.subplot(2,2,1)
plt.hist(data.no_of_children,color='brown')
plt.xlabel("No of Children")
plt.ylabel("Count")
plt.title("How many children do they have?")
plt.subplot(2,2,2)
plt.hist(data[data.credit_card_default == 1].no_of_children,color='brown')
plt.xlabel("No of Children")
plt.ylabel("Count")
plt.title("How many children do defaulters have?")
plt.subplot(2,2,3)
plt.hist(data[data.credit_card_default == 0].no_of_children,color='brown')
plt.xlabel("No of Children")
plt.ylabel("Count")
plt.title("How many children do defaulters have?")
plt.tight_layout()

<p><b>Inference : </b>The distribution stays the <b>same</b> among both classes with negligible variations.</p>

<h1>How much do they earn?</h1>


In [ ]:
plt.figure(figsize=(10,9))
plt.subplot(2,2,1)
plt.hist(data.net_yearly_income,color='aqua')
plt.xlabel("Income group")
plt.ylabel("Count")
plt.title("How much do they earn?")
plt.subplot(2,2,2)
plt.hist(data[data.credit_card_default == 1].net_yearly_income,color='aqua')
plt.xlabel("Income group")
plt.ylabel("Count")
plt.title("How much do defaulters earn?")
plt.subplot(2,2,3)
plt.hist(data[data.credit_card_default == 0].net_yearly_income,color='aqua')
plt.xlabel("Income group")
plt.ylabel("Count")
plt.title("How much do non-defaulters earn?")
plt.tight_layout()

<p><b>Inference : </b>Counter Intuitively, <b>Defaulters</b> earn in higher amount in comparison with <b>Non-defaulters</b></p>

<h1>How long is their work experience?</h1>

In [ ]:
plt.figure(figsize=(10,9))
plt.subplot(2,2,1)
plt.hist(data.employment_years,color='blue')
plt.xlabel("Time Span")
plt.ylabel("Count")
plt.title("How long have they been working?")
plt.subplot(2,2,2)
plt.hist(data[data.credit_card_default == 1].employment_years,color='blue')
plt.xlabel("Time Span")
plt.ylabel("Count")
plt.title("How long have the defaulters been working?")
plt.subplot(2,2,3)
plt.hist(data[data.credit_card_default == 0].employment_years,color='blue')
plt.xlabel("Time Span")
plt.ylabel("Count")
plt.title("How long have the non-defaulters been working?")
plt.tight_layout()

<p><b>Inference : </b>Somewhat similar distribution with <b>Moderate Work Experience</b> higher among defaulters.</p>

<h1>What is their work?</h1>

In [ ]:
shuffle(colors)
plt.figure(figsize=(15,12))
plt.subplot(2,2,1)
un, count = np.unique(data.occupation_type.dropna(), return_counts=True)
plt.barh(un, count, color=colors)
plt.xlabel("Occupation Type")
plt.ylabel("Count")
plt.title("What profession do they do?")

plt.subplot(2,2,2)
un, count = np.unique(data[data.credit_card_default == 1].occupation_type.dropna(), return_counts=True)
plt.barh(un, count, color=colors)
plt.xlabel("Occupation Type")
plt.ylabel("Count")
plt.title("What profession do defaulters do?")

plt.subplot(2,2,3)
un, count = np.unique(data[data.credit_card_default == 0].occupation_type.dropna(), return_counts=True)
plt.barh(un, count, color=colors)
plt.xlabel("Occupation Type")
plt.ylabel("Count")
plt.title("What profession do non-defaulters do?")
plt.tight_layout()

<p><b>Inference : </b>Comparatively, <b>Defaulters</b> have <b>Low-income</b> jobs compared to <b>Non-Defaulters</b>.</p>

<h1>How big is their family</h1>

In [ ]:
plt.figure(figsize=(10,9))
plt.subplot(2,2,1)
plt.hist(data.total_family_members,color='seagreen')
plt.xlabel("Family Member Count")
plt.ylabel("Count")
plt.title("How big family do they have?")
plt.subplot(2,2,2)
plt.hist(data[data.credit_card_default == 1].total_family_members,color='seagreen')
plt.xlabel("Family Member Count")
plt.ylabel("Count")
plt.title("How big family do defaulters have?")
plt.subplot(2,2,3)
plt.hist(data[data.credit_card_default == 0].total_family_members,color='seagreen')
plt.xlabel("Family Member Count")
plt.ylabel("Count")
plt.title("How big family do non-defaulters have?")
plt.tight_layout()

<p><b>Inference : </b>Similar distributions with neglible variations.</p>

<h1>Are they a migrant?</h1>

In [ ]:
shuffle(colors)
plt.figure(figsize=(10,9))
plt.subplot(2,2,1)
un, count = np.unique(data.migrant_worker.dropna(), return_counts=True)
un = [str(int(i)) for i in un]
plt.bar(un, count, color = colors)
plt.xlabel("Migration Status")
plt.ylabel("Count")
plt.title("Are they an outsider?")
plt.subplot(2,2,2)
un, count = np.unique(data[data.credit_card_default == 0].migrant_worker.dropna(), return_counts=True)
un = [str(int(i)) for i in un]
plt.bar(un, count, color = colors)
plt.xlabel("Migration Status")
plt.ylabel("Count")
plt.title("Are they non-defaulters an outsider?")
plt.subplot(2,2,3)
un, count = np.unique(data[data.credit_card_default == 1].migrant_worker.dropna(), return_counts=True)
un = [str(int(i)) for i in un]
plt.bar(un, count, color = colors)
plt.xlabel("Migration Status")
plt.ylabel("Count")
plt.title("Are they defaulters an outsider?")
plt.tight_layout()

<p><b>Inference : </b>There is a small variation in the migration status for <b>Defaulters</b>, where there are comparatively more <b>Outsiders</b>.</p>

<h1>How much is their debt?</h1>

In [ ]:
plt.figure(figsize=(15,9))
plt.subplot(2,2,1)
plt.hist(data.yearly_debt_payments, color="lightblue")
plt.xlabel("Debt Amount Range")
plt.ylabel("Count")
plt.title("How much debt do they have?")
plt.subplot(2,2,2)
plt.hist(data[data.credit_card_default == 1].yearly_debt_payments, color="lightblue")
plt.xlabel("Debt Amount Range")
plt.ylabel("Count")
plt.title("How much debt do defaulters have?")
plt.subplot(2,2,3)
plt.hist(data[data.credit_card_default == 0].yearly_debt_payments, color="lightblue")
plt.xlabel("Debt Amount Range")
plt.ylabel("Count")
plt.title("How much debt do non-defaulters have?")
plt.tight_layout()

<p><b>Inference : </b>The debt status of both classes are similar with negligible variation.</p>

<h1>What is their credit limit?</h1>

In [ ]:
plt.figure(figsize=(20,9))
plt.subplot(2,2,1)
plt.ticklabel_format(useOffset=False)
plt.hist(data.credit_limit, color="skyblue")
plt.xlabel("Available Credit Limit Range")
plt.ylabel("Count")
plt.title("How much credit limit do they have?")

plt.subplot(2,2,2)
plt.ticklabel_format(useOffset=False)
plt.hist(data[data.credit_card_default == 1].credit_limit, color="skyblue")
plt.xlabel("Available Credit Limit Range")
plt.ylabel("Count")
plt.title("How much credit limit do defaulters have?")

plt.subplot(2,2,3)
plt.ticklabel_format(useOffset=False)
plt.hist(data[data.credit_card_default == 0].credit_limit, color="skyblue")
plt.xlabel("Available Credit Limit Range")
plt.ylabel("Count")
plt.title("How much credit limit do non-defaulters have?")
plt.tight_layout()

<p><b>Inference : </b><b>Defaulters</b> have a <b>higher</b> credit limit in comparison with <b>Non-Defaulters</b> </p>

<h1>How much credit limit they used?</h1>

In [ ]:
plt.figure(figsize=(10,9))
plt.subplot(2,2,1)
plt.hist(data['credit_limit_used(%)'], color="darkgreen")
plt.xlabel("Used Credit Limit Range")
plt.ylabel("Count")
plt.title("How much credit limit have they used?")
plt.subplot(2,2,2)
plt.hist(data[data.credit_card_default == 1]['credit_limit_used(%)'], color="darkgreen")
plt.xlabel("Used Credit Limit Range")
plt.ylabel("Count")
plt.title("How much credit limit have defaulters used?")
plt.subplot(2,2,3)
plt.hist(data[data.credit_card_default == 0]['credit_limit_used(%)'], color="darkgreen")
plt.xlabel("Used Credit Limit Range")
plt.ylabel("Count")
plt.title("How much credit limit have non-defaulters used?")
plt.tight_layout()

<p><b>Inference : </b><b>Defaulters</b> have used a major part of their credit limit- <b>70%</b> or more in comparison with <b>Non-Defaulters</b> where there is a minimum usage of <b>0%</b>.</p>

<h1>What is their credit score?</h1>

In [ ]:
shuffle(colors)
plt.figure(figsize=(10,9))
plt.subplot(2,2,1)
plt.hist(data.credit_score, color=colors[0])
plt.xlabel("Credit Score Range")
plt.ylabel("Count")
plt.title("How much credit score do they have?")
plt.subplot(2,2,2)
plt.hist(data[data.credit_card_default == 1].credit_score, color=colors[0])
plt.xlabel("Credit Score Range")
plt.ylabel("Count")
plt.title("How much credit score do defaulters have?")
plt.subplot(2,2,3)
plt.hist(data[data.credit_card_default == 0].credit_score, color=colors[0])
plt.xlabel("Credit Score Range")
plt.ylabel("Count")
plt.title("How much credit score do non-defaulters have?")
plt.tight_layout()

<p><b>Inference : </b>The credit score of <b>Defaulters</b> range from <b>500 - 700</b> while for <b>Non-Defaulters</b> range from <b>650 - 950</b>.</p>

<h1>How much time did they default before?</h1>

In [ ]:
shuffle(colors)
plt.figure(figsize=(10,9))
plt.subplot(2,2,1)
un, count = np.unique(data.prev_defaults, return_counts=True)
un = [str(int(i)) for i in un]
plt.bar(un, count, color=colors)
plt.xlabel("Previous Default Count")
plt.ylabel("Count")
plt.title("How many times have they defaulted before?")
plt.subplot(2,2,2)
un, count = np.unique(data[data.credit_card_default == 1].prev_defaults, return_counts=True)
un = [str(int(i)) for i in un]
plt.bar(un, count, color=colors)
plt.xlabel("Previous Default Count")
plt.ylabel("Count")
plt.title("How many times have the defaulters defaulted before?")
plt.subplot(2,2,3)
un, count = np.unique(data[data.credit_card_default == 0].prev_defaults, return_counts=True)
un = [str(int(i)) for i in un]
plt.bar(un, count, color=colors)
plt.xlabel("Previous Default Count")
plt.ylabel("Count")
plt.title("How many times have the non-defaulters defaulted before?")
plt.tight_layout()

<p><b>Inference : </b>Majority of the <b>Defaulters</b> have defaulted <b>Atleast</b> previously.</p>

<h1>Have they defaulted recently?</h1>

In [ ]:
shuffle(colors)
plt.figure(figsize=(10,9))
plt.subplot(2,2,1)
un, count = np.unique(data.default_in_last_6months, return_counts=True)
un = [str(int(i)) for i in un]
plt.bar(un, count, color=colors)
plt.xlabel("Recent Default Count")
plt.ylabel("Count")
plt.title("Have they defaulted recently?")
plt.subplot(2,2,2)
un, count = np.unique(data[data.credit_card_default == 1].default_in_last_6months, return_counts=True)
un = [str(int(i)) for i in un]
plt.bar(un, count, color=colors)
plt.xlabel("Recent Default Count")
plt.ylabel("Count")
plt.title("Have the defaulters defaulted recently?")
plt.subplot(2,2,3)
un, count = np.unique(data[data.credit_card_default == 0].default_in_last_6months, return_counts=True)
un = [str(int(i)) for i in un]
plt.bar(un, count, color=colors)
plt.xlabel("Recent Default Count")
plt.ylabel("Count")
plt.title("Have the non-defaulters defaulted recently?")
plt.tight_layout()

<p><b>Inference : </b>Majority of the defaulters have <b>Defaulted</b> in the last 6 months.</p>

<h1>How much Profit / Loss occurs to them?</h1>

In [ ]:
plt.figure(figsize=(20,9))
plt.subplot(2,2,1)
shuffle(colors)
plt.xlabel("In hand profit range")
plt.ylabel("Count")
plt.title("In what range does loss occur to defaulters?")
plt.ticklabel_format(useOffset=False)
plt.hist([i for i in data[data.credit_card_default == 1].in_hand_balance if i < 0], color=colors[0])

plt.subplot(2,2,2)
plt.xlabel("In hand profit range")
plt.ylabel("Count")
plt.title("In what range does profit occur to defaulters?")
plt.ticklabel_format(useOffset=False)
plt.hist([i for i in data[data.credit_card_default == 1].in_hand_balance if i >= 0], color=colors[0])

plt.subplot(2,2,3)
plt.xlabel("In hand profit range")
plt.ylabel("Count")
plt.title("In what range does loss occur to non-defaulters?")
plt.ticklabel_format(useOffset=False)
plt.hist([i for i in data[data.credit_card_default == 0].in_hand_balance if i < 0], color=colors[0])

plt.subplot(2,2,4)
plt.xlabel("In hand profit range")
plt.ylabel("Count")
plt.title("In what range does profit occur to non-defaulters?")
plt.ticklabel_format(useOffset=False)
plt.hist([i for i in data[data.credit_card_default == 0].in_hand_balance if i >= 0], color=colors[0])

plt.tight_layout()

<p><b>Inference : </b>For <b>Defaulters</b>, the loss seems to be a higher number in comparison with <b>Non-Defaulters</b>, but counter intuitively, In hand profit appears <b>Higher</b> for <b>Defaulters</b>.</p>

<h1>How many are saving?</h1>

In [ ]:
shuffle(colors)
plt.figure(figsize=(10,9))
plt.subplot(2,2,1)
un, count = np.unique(data.in_profit, return_counts=True)
plt.bar(un, count, color=colors)
plt.title("How many are able to save money?")
plt.xlabel("Profit status")
plt.ylabel("Count")
plt.subplot(2,2,2)
un, count = np.unique(data[data.credit_card_default == 1].in_profit, return_counts=True)
plt.bar(un, count, color=colors)
plt.title("How many defaulters are able to save money?")
plt.xlabel("Profit status")
plt.ylabel("Count")
plt.subplot(2,2,3)
un, count = np.unique(data[data.credit_card_default == 0].in_profit, return_counts=True)
plt.bar(un, count, color=colors)
plt.title("How many non-defaulters are able to save money?")
plt.xlabel("Profit status")
plt.ylabel("Count")
plt.tight_layout()

<p><b>Inference : </b> All of them are making a <b>profit</b>, with negligibly less <b>Loss</b> makers.</p>

<h1>How much do they earn?</h1>

In [ ]:
shuffle(colors)
plt.figure(figsize=(15,9))
plt.subplot(2,2,1)
plt.hist(data.total_income_received, color=colors[0])
plt.xlabel("Income Range")
plt.ylabel("Count")
plt.title("How much do they earn?")

plt.subplot(2,2,2)
plt.hist(data[data.credit_card_default == 1].total_income_received, color=colors[0])
plt.xlabel("Income Range")
plt.ylabel("Count")
plt.title("How much do defaulters earn?")

plt.subplot(2,2,3)
plt.hist(data[data.credit_card_default == 0].total_income_received, color=colors[0])
plt.xlabel("Income Range")
plt.ylabel("Count")
plt.title("How much do non-defaulters earn?")
plt.tight_layout()

<p><b>Inference : </b><b>Defaulters</b> are having a slightly higher income range in comparison with <b>Non-Defaulters</b>.</p>

<h1>How many defaulted?</h1>

In [ ]:
shuffle(colors)
un, count = np.unique(data.credit_card_default, return_counts=True)
un = [str(int(i)) for i in un]
plt.bar(un, count, color = colors)
plt.xlabel("Default")
plt.ylabel("Count")
plt.title("How many were defaulters?")
plt.show()

<p><b>Inference : </b>Majority of them were <b>Non-Defaulters</b>.</p>

# Leakage-safe credit card default modeling

**Target:** `credit_card_default` (customer default risk). This dataset does not contain individual transaction fraud labels, so the reported results describe default prediction, not transaction fraud detection. All model preprocessing is fit on the training partition only. A separate validation partition selects the model; the untouched test partition is used once for final reporting. Install `imbalanced-learn` and optionally `xgboost` in the notebook environment before running.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

# Point this to the labeled CSV if running outside Kaggle.
DATA_PATH = Path('/kaggle/input/amex-competition/dataset/train.csv')
raw = pd.read_csv(DATA_PATH)
assert 'credit_card_default' in raw.columns
assert set(raw['credit_card_default'].dropna().unique()) == {0, 1}, 'Expected binary 0/1 labels'
raw['credit_card_default'].value_counts(dropna=False)


In [ ]:
def add_features(frame):
    frame = frame.copy()
    # Row-wise arithmetic uses no population statistics or target values.
    frame['in_hand_balance'] = frame['net_yearly_income'] - frame['yearly_debt_payments']
    frame['total_income_received'] = frame['net_yearly_income'] * frame['no_of_days_employed']
    frame['employment_years'] = frame['no_of_days_employed'] / 365.0
    return frame

X = add_features(raw.drop(columns=['credit_card_default', 'customer_id', 'name'], errors='ignore'))
y = raw['credit_card_default'].astype(int)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=42)
print('Train, validation, test sizes:', len(y_train), len(y_val), len(y_test))
print('Test positive prevalence:', round(y_test.mean(), 4))


In [ ]:
numeric = X.select_dtypes(include=np.number).columns.tolist()
categorical = X.columns.difference(numeric).tolist()
preprocess = ColumnTransformer([
    ('numeric', SkPipeline([('impute', SimpleImputer(strategy='median')),
                            ('scale', StandardScaler())]), numeric),
    ('categorical', SkPipeline([('impute', SimpleImputer(strategy='most_frequent')),
                                ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical),
], sparse_threshold=0)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
}
try:
    from xgboost import XGBClassifier
    models['XGBoost'] = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                                    eval_metric='logloss', n_jobs=-1, random_state=42)
except ImportError:
    print('xgboost is unavailable; comparing Logistic Regression and Random Forest.')

# Each fit learns imputation, scaling, one-hot categories, and SMOTE neighbors
# from X_train alone. Neither validation nor test rows are resampled.
fitted = {}
rows = []
for name, estimator in models.items():
    pipeline = Pipeline([('preprocess', preprocess),
                         ('smote', SMOTE(random_state=42)),
                         ('model', estimator)])
    pipeline.fit(X_train, y_train)
    fitted[name] = pipeline
    scores = pipeline.predict_proba(X_val)[:, 1]
    predictions = (scores >= 0.5).astype(int)
    rows.append({'model': name, 'validation_pr_auc': average_precision_score(y_val, scores),
                 'validation_roc_auc': roc_auc_score(y_val, scores),
                 'validation_f1': f1_score(y_val, predictions, zero_division=0)})
comparison = pd.DataFrame(rows).sort_values('validation_pr_auc', ascending=False)
display(comparison)


## Final evaluation

Select by validation PR-AUC, then evaluate that frozen fitted pipeline on the natural-prevalence test set. The 0.5 decision threshold is fixed in advance. PR-AUC here is average precision. False-positive rate is FP / (FP + TN). Do not use the test results to tune features, model, or threshold.


In [ ]:
best_name = comparison.iloc[0]['model']
best = fitted[best_name]
test_scores = best.predict_proba(X_test)[:, 1]
y_pred = (test_scores >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
results = pd.Series({
    'model': best_name,
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred, zero_division=0),
    'recall': recall_score(y_test, y_pred, zero_division=0),
    'f1': f1_score(y_test, y_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, test_scores),
    'pr_auc': average_precision_score(y_test, test_scores),
    'false_positive_rate': fp / (fp + tn),
    'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
})
display(results.to_frame('untouched test'))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=[0, 1],
                                         display_labels=['No default', 'Default'])
plt.show()


## Explain the predictions

Permutation importance measures the change in PR-AUC when one **original input column** is shuffled on validation data. It is a global association, not a causal explanation or a per-customer reason. For a specific customer, the risk score below can be inspected alongside their inputs; do not interpret global importance as an individual attribution.


In [ ]:
importance = permutation_importance(best, X_val, y_val, scoring='average_precision',
                                    n_repeats=3, random_state=42, n_jobs=1)
feature_importance = pd.Series(importance.importances_mean, index=X_val.columns).sort_values(ascending=False)
display(feature_importance.head(15).to_frame('validation PR-AUC decrease'))
feature_importance.head(15).sort_values().plot.barh(title='Permutation importance (validation PR-AUC)')
plt.tight_layout()
plt.show()
example = X_test.iloc[[0]]
print('Example held-out customer predicted default probability:', round(float(best.predict_proba(example)[0, 1]), 4))
display(example.T.rename(columns={example.index[0]: 'input value'}))


### Reproducibility and claims

Run all cells with the labeled Kaggle dataset available. The matching Kaggle dataset is `hotsonhonet/amex-competition`, file `dataset/train.csv`. The repository does not contain the dataset; saved old outputs have been removed because they were computed with leakage. Copy resume metrics only from the freshly executed **untouched test** results above. The earlier `/content/test.csv` prediction section was removed because it independently fit encoders and did not use the training preprocessing pipeline.


### Verified run on the matching Kaggle dataset

Source: [`hotsonhonet/amex-competition`](https://www.kaggle.com/datasets/hotsonhonet/amex-competition), `dataset/train.csv` (45,528 labeled customers; 8.12% positives in the untouched test set). The modeling cells were executed on the downloaded dataset with the stated random seed and a fixed 0.5 threshold. The comparison selected XGBoost by validation PR-AUC (0.9483, versus Random Forest 0.9474 and Logistic Regression 0.9390).

| Untouched test metric | XGBoost |
| --- | ---: |
| Accuracy | 0.9616 |
| Precision | 0.6912 |
| Recall | 0.9513 |
| F1 | 0.8007 |
| ROC-AUC | 0.9951 |
| PR-AUC (average precision) | 0.9559 |
| False-positive rate | 0.0375 |
| TN / FP / FN / TP | 8053 / 314 / 36 / 703 |

These are default-risk results from this dataset and split, not transaction-fraud results or a Kaggle competition score. Re-execute the notebook to regenerate values in your own environment.
